In [7]:
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.sparse.linalg import LinearOperator, cg
Nx, Ny = 201, 201

EPS0, MU0 = 1,1
#EPS0, MU0 = 8.854e-12, 4 * np.pi * 1e-7
C0 = 1 / np.sqrt(EPS0 * MU0)

sigma = 0.03*1e-1
f_max = (5/sigma)/(2*np.pi)

lambda_min = C0 / f_max       # minimum wavelength
dx = lambda_min / 50         # spatial step
dy = dx
CFL = 1
dt = CFL / (C0*np.sqrt(1/dx**2 + 1/dy**2)) 
N = Nx * Ny


Nt = 200



# Center of the grid
source_x, source_y = Nx // 2, Ny // 2
source_index = source_x * Ny + source_y  # Flattened index for (source_x, source_y)

def get_source_value(t, dt):
    # Gaussian pulse
    t0 = 20 * dt  # Peak time
    sigma = 5 * dt # Width
    return 1e6 *np.exp(-((t - t0)**2) / (2 * sigma**2))


def plot_matrix(matrix):
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap='RdBu_r', aspect='auto')
    plt.colorbar(label='Value')
    plt.title('d_x Matrix Visualization')
    plt.xlabel('Column Index')
    plt.ylabel('Row Index')
    plt.show()

Ez = np.zeros((Nx,Ny))
Hx = Ez.copy()
Hy = Ez.copy()

Ezflat = Ez.ravel()
Hxflat = Hx.ravel()
Hyflat = Hy.ravel()

fields = np.concatenate([Hxflat, Hyflat, Ezflat])


d_x_circulant = sparse.diags([-1, 1], [0, 1], shape=(Nx, Nx), format='lil')
d_x_circulant[-1, 0] = 1  # Wrap-around: last row connects to first column


d_y_circulant = sparse.diags([-1, 1], [0, 1], shape=(Ny, Ny), format='lil')
d_y_circulant[-1, 0] = 1


Ax_circulant = sparse.diags([0.5, 0.5], [0, 1], shape=(Nx, Nx), format='lil')
Ax_circulant[-1, 0] = 0.5  # Wrap-around


Ay_circulant = sparse.diags([0.5, 0.5], [0, 1], shape=(Ny, Ny), format='lil')
Ay_circulant[-1, 0] = 0.5


I_x = sparse.eye(Nx,format='csc')
I_y = sparse.eye(Ny,format='csc')

diag_step_x = sparse.diags([dx],shape=(Nx,Nx),format='csc')
diag_step_y = sparse.diags([dy],shape=(Ny,Ny),format='csc')

dx_array = dx* np.ones((Nx,))

diag_step_x = sparse.diags(dx_array, format='csc')
diag_step_y = sparse.diags(dx_array,format='csc')

diag_step_x_inverse = sparse.diags(1.0 / dx_array, format='csc')
diag_step_y_inverse = sparse.diags(1.0 / dx_array, format='csc')
diag_step_xx = sparse.kron(diag_step_x,I_y)
diag_step_xx_inverse = sparse.kron(diag_step_x_inverse,I_y)
diag_step_yy = sparse.kron(I_x,diag_step_y)
diag_step_yy_inverse = sparse.kron(I_x,diag_step_y_inverse)

diag_step_x_dual = sparse.diags(dx_array, format='csc')
diag_step_y_dual = sparse.diags(dx_array,format='csc')

diag_step_x_inverse_dual = sparse.diags(1.0 / dx_array, format='csc')
diag_step_y_inverse_dual = sparse.diags(1.0 / dx_array, format='csc')
diag_step_xx_dual = sparse.kron(diag_step_x,I_y)
diag_step_xx_inverse_dual = sparse.kron(diag_step_x_inverse,I_y)
diag_step_yy_dual = sparse.kron(I_x,diag_step_y)
diag_step_yy_inverse_dual = sparse.kron(I_x,diag_step_y_inverse)

hodge_mu_xx = MU0 * sparse.kron(I_x, Ay_circulant)
hodge_mu_yy = MU0 * sparse.kron(Ax_circulant, I_y)
hodge_eps_zz = EPS0 * sparse.kron(Ax_circulant, Ay_circulant)

hodge_c_xz = sparse.kron(I_x, diag_step_y_inverse @ d_y_circulant)
hodge_c_yz = sparse.kron(-diag_step_x_inverse @ d_x_circulant, I_y)

hodge_c_zx = sparse.kron(-Ax_circulant, diag_step_y_inverse @ d_y_circulant)
hodge_c_zy = sparse.kron(diag_step_x_inverse @ d_x_circulant, Ay_circulant)


Mleft = sparse.bmat([[hodge_mu_xx/dt,None,-hodge_c_xz/2],
                     [None,hodge_mu_yy/dt,-hodge_c_yz/2],
                     [hodge_c_zx/2,hodge_c_zy/2,hodge_eps_zz/dt]])

Mright = sparse.bmat([[hodge_mu_xx/dt,None,hodge_c_xz/2],
                     [None,hodge_mu_yy/dt,hodge_c_yz/2],
                     [-hodge_c_zx/2,-hodge_c_zy/2,hodge_eps_zz/dt]])


Mleft_solver =sparse.linalg.factorized(Mleft)

def update(fields,current_time):
    Hx = fields[:N].copy()
    Hy = fields[N:2*N].copy()
    Ez = fields[2*N:].copy()

    
   

    
    fields_old = np.concatenate([Hx, Hy, Ez])
    rhs = Mright @ fields_old

    # inject source into RHS (Ez block)
    rhs[2*N + source_index] += get_source_value(current_time, dt)

    # Solve for next step
    fields_new = Mleft_solver(rhs)

    return fields_new

# ============================================
# RUN SIMULATION
# ============================================
print("Running simulation...")
snapshots = {}
snap_interval = 10  # Save every 10 frames

for n in range(Nt):
    if n % 50 == 0:
        print(f"Step {n}/{Nt}")
    
    fields = update(fields, n * dt)
    
    if n % snap_interval == 0:
        Ez = fields[2*N:].reshape((Nx, Ny))
        snapshots[n] = Ez.copy()

print("Simulation complete!")

# ============================================
# CREATE ANIMATION
# ============================================
fig, ax = plt.subplots(figsize=(8, 7))

# Find global min/max for consistent colormap
all_ez = np.array(list(snapshots.values()))
vmin, vmax = all_ez.min(), all_ez.max()
vmax_abs = max(abs(vmin), abs(vmax))
vmin, vmax = -vmax_abs, vmax_abs

# First frame
snap_list = sorted(snapshots.keys())
im = ax.imshow(snapshots[snap_list[0]], cmap='RdBu', origin='lower',
               vmin=vmin, vmax=vmax, extent=[0, Ny*dx, 0, Nx*dx])
plt.colorbar(im, ax=ax, label='Ez Field')
title = ax.set_title(f"Time = 0.00 s")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")

def animate(frame_idx):
    n = snap_list[frame_idx]
    Ez_data = snapshots[n]
    im.set_data(Ez_data)
    t = n * dt
    title.set_text(f"Time = {t:.2e} s")
    return [im, title]

anim = animation.FuncAnimation(
    fig, animate, frames=len(snap_list),
    interval=50, blit=True, repeat=True
)

# Display animation
from IPython.display import display, HTML
display(HTML(anim.to_jshtml()))
plt.close()


c:\Users\Olivier\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:597: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve


Running simulation...
Step 0/200
Step 50/200
Step 100/200
Step 150/200
Simulation complete!


In [ ]:
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Rectangle
from IPython.display import display, HTML

# ── Fundamentele constanten ───────────────────────────────────────────────────
EPS0, MU0 = 8.854187817e-12, 4 * np.pi * 1e-7
C0 = 1 / np.sqrt(EPS0 * MU0)

# ── Domein ────────────────────────────────────────────────────────────────────
domain_width  = 0.5*e8
domain_height = 0.5*e8

# ── Frequentie / puls ─────────────────────────────────────────────────────────
sigma_t = 0.1e-1
f_max   = (5 / sigma_t) / (2 * np.pi)

# ── Rooster ───────────────────────────────────────────────────────────────────
lambda_min = C0 / f_max
dx = lambda_min / 15
dy = dx
CFL = 0.999999
dt  = 2*1.184768e-03

Nx = round(domain_width  / dx)
Ny = round(domain_height / dy)
N  = Nx * Ny
Nt = round(20 * sigma_t / dt)

print(f"Grid size: {Nx} x {Ny}, Time steps: {Nt}")
print(f"dx = {dx:.4f} m, dt = {dt:.6e} s")
print(f"Lambda_min = {lambda_min:.4f} m, points per wavelength = {lambda_min/dx:.1f}")

# ── Bron ──────────────────────────────────────────────────────────────────────
ps_x = int(domain_width  / 2.5 / dx)
ps_y = int(domain_height / 2.3 / dy)
source_index = ps_x * Ny + ps_y

J0 = 10
tc = 5 * sigma_t

def get_source_value(t, dt):
    return J0 * np.exp(-(t - tc)**2 / (2 * sigma_t**2))

# ── Observatiepunt ────────────────────────────────────────────────────────────
po_x = int(domain_width  / 2.5 / dx)
po_y = int(domain_height / 2.1 / dy)

# ── PML-profiel ───────────────────────────────────────────────────────────────
d_PML   = domain_width / 4
N_PML   = int(d_PML / dx)
m       = 3.5
eta0    = np.sqrt(MU0 / EPS0)

sigma_max   = (m + 1) / (2 * eta0 * d_PML) * 11*0
sigma_m_max = sigma_max * MU0 / EPS0

sigma_pml   = sigma_max   * (np.arange(N_PML + 1) / N_PML) ** m
sigma_m_pml = sigma_m_max * (np.arange(N_PML + 1) / N_PML) ** m

print(f"N_PML = {N_PML}")

def sigma_PML(component):
    """
    Retourneert een 2D array (shape passend bij component) met PML-waarden,
    daarna te ravel()-en naar een 1D diagonaal.

    component  shape         beschrijving
    ---------  -----------   --------------------------------
    'Ezx'      (Nx, Ny)      electric, gesplitst in x
    'Ezy'      (Nx, Ny)      electric, gesplitst in y
    'Hx'       (Nx, Ny)      magnetic Hx
    'Hy'       (Nx, Ny)      magnetic Hy
    """
    arr = np.zeros((Nx, Ny))

    if component == 'Ezx':
        # PML aan linker- en rechterrand (x-richting)
        arr[:N_PML + 1, :] = sigma_pml[::-1, np.newaxis]
        arr[-N_PML - 1:, :] = sigma_pml[:, np.newaxis]

    elif component == 'Ezy':
        # PML aan boven- en onderrand (y-richting)
        arr[:, :N_PML + 1]  = sigma_pml[np.newaxis, ::-1]
        arr[:, -N_PML - 1:] = sigma_pml[np.newaxis, :]

    elif component == 'Hx':
        arr[:N_PML + 1, :]  = sigma_m_pml[::-1, np.newaxis]
        arr[-N_PML - 1:, :] = sigma_m_pml[:, np.newaxis]

    elif component == 'Hy':
        arr[:, :N_PML + 1]  = sigma_m_pml[np.newaxis, ::-1]
        arr[:, -N_PML - 1:] = sigma_m_pml[np.newaxis, :]

    return arr

# ── PML diagonaalmatrices (sparse, N×N) ───────────────────────────────────────
sigma_m_x = sparse.diags(sigma_PML('Hx').ravel(),  format='csc')
sigma_m_y = sparse.diags(sigma_PML('Hy').ravel(),  format='csc')
sigma_e_x = sparse.diags(sigma_PML('Ezx').ravel(), format='csc')
sigma_e_y = sparse.diags(sigma_PML('Ezy').ravel(), format='csc')
print("PML matrices created")

# ── 1-D circulante operatoren ─────────────────────────────────────────────────
d_x = sparse.diags([-1, 1], [0, 1], shape=(Nx, Nx), format='lil')
d_x[-1, 0] = 1
d_x = d_x.tocsc()

d_y = sparse.diags([-1, 1], [0, 1], shape=(Ny, Ny), format='lil')
d_y[-1, 0] = 1
d_y = d_y.tocsc()

Ax = sparse.diags([0.5, 0.5], [0, 1], shape=(Nx, Nx), format='lil')
Ax[-1, 0] = 0.5
Ax = Ax.tocsc()

Ay = sparse.diags([0.5, 0.5], [0, 1], shape=(Ny, Ny), format='lil')
Ay[-1, 0] = 0.5
Ay = Ay.tocsc()

I_x = sparse.eye(Nx, format='csc')
I_y = sparse.eye(Ny, format='csc')

dx_arr = dx * np.ones(Nx)
dy_arr = dy * np.ones(Ny)
diag_step_x_inv = sparse.diags(1.0 / dx_arr, format='csc')
diag_step_y_inv = sparse.diags(1.0 / dy_arr, format='csc')

# ── Hodge-operatoren ──────────────────────────────────────────────────────────
hodge_mu_xx  = MU0  * sparse.kron(I_x, Ay, format='csc')
hodge_mu_yy  = MU0  * sparse.kron(Ax,  I_y, format='csc')
hodge_eps_zz = EPS0 * sparse.kron(Ax,  Ay,  format='csc')

hodge_c_xz = sparse.kron(I_x,                      diag_step_y_inv @ d_y, format='csc')
hodge_c_yz = sparse.kron(-diag_step_x_inv @ d_x,   I_y,                   format='csc')
hodge_c_zx = sparse.kron(-Ax,                       diag_step_y_inv @ d_y, format='csc')
hodge_c_zy = sparse.kron( diag_step_x_inv @ d_x,   Ay,                    format='csc')

Z = None  # nulblok voor sparse.bmat

# ── Systeemmatrices [hx, hy, ezx, ezy] ──────────────────────────────────────
Mleft = sparse.bmat([
    [hodge_mu_xx/dt  + sigma_m_x/2,  Z,                               -hodge_c_xz/4,                        -hodge_c_xz/4                     ],
    [Z,                               hodge_mu_yy/dt  + sigma_m_y/2,  -hodge_c_yz/4,                        -hodge_c_yz/4                       ],
    [Z,                               hodge_c_zy/2,                   hodge_eps_zz/dt + sigma_e_x/2,       Z                                  ],
    [hodge_c_zx/2,                    Z,                               Z,                                   hodge_eps_zz/dt + sigma_e_y/2      ],
], format='csc')

Mright = sparse.bmat([
    [hodge_mu_xx/dt  - sigma_m_x/2,  Z,                               hodge_c_xz/4,                       hodge_c_xz/4                      ],
    [Z,                               hodge_mu_yy/dt  - sigma_m_y/2,  hodge_c_yz/4,                       hodge_c_yz/4                        ],
    [Z,                               -hodge_c_zy/2,                  hodge_eps_zz/dt - sigma_e_x/2,       Z                                  ],
    [-hodge_c_zx/2,                   Z,                               Z,                                   hodge_eps_zz/dt - sigma_e_y/2      ],
], format='csc')

print("LU-factorisatie...")
Mleft_solver = sparse.linalg.factorized(Mleft)
print("Klaar.")

# ── Beginvelden [hx | hy | ezx | ezy] ───────────────────────────────────────
fields = np.zeros(4 * N)

def update(fields, current_time):
    rhs = Mright @ fields
    rhs[2*N + source_index] += get_source_value(current_time, dt)
    return Mleft_solver(rhs)

# ── Tijdlus ───────────────────────────────────────────────────────────────────
print("Simulatie gestart...")
snapshots     = {}
snap_interval = 10

for n in range(Nt):
    if n % 50 == 0:
        print(f"  Stap {n}/{Nt}")
    fields = update(fields, n * dt)
    if n % snap_interval == 0:
        ezx = fields[2*N:3*N].reshape((Nx, Ny))
        ezy = fields[3*N:4*N].reshape((Nx, Ny))
        snapshots[n] = (ezx + ezy).copy()

print("Simulatie klaar!")

# ── Animatie ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

snap_list = sorted(snapshots.keys())
all_ez = np.array(list(snapshots.values()))
vmax_abs = max(abs(all_ez.min()), abs(all_ez.max())) or 1.0

# Convert from meters to mm for display
x_mm = np.arange(Nx) * dx * 1e3
y_mm = np.arange(Ny) * dy * 1e3

im = ax.imshow(snapshots[snap_list[0]].T, cmap='RdBu', origin='lower',
               vmin=-vmax_abs, vmax=vmax_abs,
               extent=[0, Nx*dx*1e3, 0, Ny*dy*1e3])
plt.colorbar(im, ax=ax, label='$E_z$ [V/m]')

# PML-grenzen
for lim in [N_PML*dx*1e3, (Nx-N_PML)*dx*1e3]:
    ax.axvline(lim, color='gray', lw=0.8, ls='--', alpha=0.6)
for lim in [N_PML*dy*1e3, (Ny-N_PML)*dy*1e3]:
    ax.axhline(lim, color='gray', lw=0.8, ls='--', alpha=0.6)

# Bron en observatiepunt
ax.plot(ps_x*dx*1e3, ps_y*dy*1e3, 'r*', ms=10, label='Bron')
ax.plot(po_x*dx*1e3, po_y*dy*1e3, 'bx', ms=10, label='Observatie')

ax.legend(loc='upper right', fontsize=10)
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')
title = ax.set_title(f"Time = 0.00 s")

def animate(frame_idx):
    n = snap_list[frame_idx]
    im.set_data(snapshots[n].T)
    title.set_text(f"Time = {n*dt:.2e} s")
    return [im, title]

anim = animation.FuncAnimation(
    fig, animate, frames=len(snap_list),
    interval=50, blit=True, repeat=True
)

display(HTML(anim.to_jshtml()))
plt.close()

# def plot_matrix(matrix):
#     plt.figure(figsize=(10, 8))
#     plt.imshow(matrix, cmap='RdBu_r', aspect='auto')
#     plt.colorbar(label='Value')
#     plt.title('d_x Matrix Visualization')
#     plt.xlabel('Column Index')
#     plt.ylabel('Row Index')
#     plt.show()
# plot_matrix(fields)

Grid size: 398 x 398, Time steps: 84
dx = 251153.5423 m, dt = 2.369536e-03 s
Lambda_min = 3767303.1347 m, points per wavelength = 15.0
N_PML = 99
PML matrices created
LU-factorisatie...
Klaar.
Simulatie gestart...
  Stap 0/84
  Stap 50/84
Simulatie klaar!


In [6]:
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Rectangle
from IPython.display import display, HTML

# ── Fundamentele constanten ───────────────────────────────────────────────────
EPS0, MU0 = 8.854187817e-12, 4 * np.pi * 1e-7
C0 = 1 / np.sqrt(EPS0 * MU0)

# ── Domein ────────────────────────────────────────────────────────────────────
domain_width  = 0.5e8
domain_height = 0.5e8

# ── Frequentie / puls ─────────────────────────────────────────────────────────
sigma_t = 0.1e-1
f_max   = (5 / sigma_t) / (2 * np.pi)

# ── Rooster ───────────────────────────────────────────────────────────────────
lambda_min = C0 / f_max
dx = lambda_min / 15
dy = dx
CFL = 0.999999
dt  = 2*1.184768e-03

Nx = round(domain_width  / dx)
Ny = round(domain_height / dy)
N  = Nx * Ny
Nt = round(40 * sigma_t / dt)

print(f"Grid size: {Nx} x {Ny}, Time steps: {Nt}")
print(f"dx = {dx:.4f} m, dt = {dt:.6e} s")
print(f"Lambda_min = {lambda_min:.4f} m, points per wavelength = {lambda_min/dx:.1f}")

# ── Bron ──────────────────────────────────────────────────────────────────────
ps_x = int(domain_width  / 2.5 / dx)
ps_y = int(domain_height / 2.3 / dy)
source_index = ps_x * Ny + ps_y

J0 = 10
tc = 5 * sigma_t

def get_source_value(t, dt):
    return J0 * np.exp(-(t - tc)**2 / (2 * sigma_t**2))

# ── Observatiepunt ────────────────────────────────────────────────────────────
po_x = int(domain_width  / 2.5 / dx)
po_y = int(domain_height / 2.1 / dy)

# ── PML-profiel ───────────────────────────────────────────────────────────────
d_PML   = domain_width / 4
N_PML   = int(d_PML / dx)
m       = 3.5
eta0    = np.sqrt(MU0 / EPS0)

sigma_max   = (m + 1) / (2 * eta0 * d_PML) * 2
sigma_m_max = sigma_max * MU0 / EPS0

sigma_pml   = sigma_max   * (np.arange(N_PML + 1) / N_PML) ** m
sigma_m_pml = sigma_m_max * (np.arange(N_PML + 1) / N_PML) ** m

print(f"N_PML = {N_PML}")

def sigma_PML(component):
    """
    Retourneert een 2D array (shape passend bij component) met PML-waarden,
    daarna te ravel()-en naar een 1D diagonaal.

    component  shape         beschrijving
    ---------  -----------   --------------------------------
    'Ezx'      (Nx, Ny)      electric, gesplitst in x
    'Ezy'      (Nx, Ny)      electric, gesplitst in y
    'Hx'       (Nx, Ny)      magnetic Hx
    'Hy'       (Nx, Ny)      magnetic Hy
    """
    arr = np.zeros((Nx, Ny))

    if component == 'Ezx':
        # PML aan linker- en rechterrand (x-richting)
        arr[:N_PML + 1, :] = sigma_pml[::-1, np.newaxis]
        arr[-N_PML - 1:, :] = sigma_pml[:, np.newaxis]

    elif component == 'Ezy':
        # PML aan boven- en onderrand (y-richting)
        arr[:, :N_PML + 1]  = sigma_pml[np.newaxis, ::-1]
        arr[:, -N_PML - 1:] = sigma_pml[np.newaxis, :]

    elif component == 'Hx':
        arr[:N_PML + 1, :]  = sigma_m_pml[::-1, np.newaxis]
        arr[-N_PML - 1:, :] = sigma_m_pml[:, np.newaxis]

    elif component == 'Hy':
        arr[:, :N_PML + 1]  = sigma_m_pml[np.newaxis, ::-1]
        arr[:, -N_PML - 1:] = sigma_m_pml[np.newaxis, :]

    return arr

# ── PML diagonaalmatrices (sparse, N×N) ───────────────────────────────────────
sigma_m_x = sparse.diags(sigma_PML('Hx').ravel(),  format='csc')
sigma_m_y = sparse.diags(sigma_PML('Hy').ravel(),  format='csc')
sigma_e_x = sparse.diags(sigma_PML('').ravel(), format='csc')
sigma_e_y = sparse.diags(sigma_PML('').ravel(), format='csc')
print("PML matrices created")
# print(sigma_e_x)
# print(sigma_e_y)

# ── 1-D circulante operatoren ─────────────────────────────────────────────────
d_x = sparse.diags([-1, 1], [0, 1], shape=(Nx, Nx), format='lil')
d_x[-1, 0] = 1
d_x = d_x.tocsc()

d_y = sparse.diags([-1, 1], [0, 1], shape=(Ny, Ny), format='lil')
d_y[-1, 0] = 1
d_y = d_y.tocsc()

Ax = sparse.diags([0.5, 0.5], [0, 1], shape=(Nx, Nx), format='lil')
Ax[-1, 0] = 0.5
Ax = Ax.tocsc()

Ay = sparse.diags([0.5, 0.5], [0, 1], shape=(Ny, Ny), format='lil')
Ay[-1, 0] = 0.5
Ay = Ay.tocsc()

I_x = sparse.eye(Nx, format='csc')
I_y = sparse.eye(Ny, format='csc')

dx_arr = dx * np.ones(Nx)
dy_arr = dy * np.ones(Ny)
diag_step_x_inv = sparse.diags(1.0 / dx_arr, format='csc')
diag_step_y_inv = sparse.diags(1.0 / dy_arr, format='csc')

# ── Hodge-operatoren ──────────────────────────────────────────────────────────
hodge_mu_xx  = MU0  * sparse.kron(I_x, Ay, format='csc')
hodge_mu_yy  = MU0  * sparse.kron(Ax,  I_y, format='csc')
hodge_eps_zz = EPS0 * sparse.kron(Ax,  Ay,  format='csc')

hodge_c_xz = sparse.kron(I_x,                      diag_step_y_inv @ d_y, format='csc')
hodge_c_yz = sparse.kron(-diag_step_x_inv @ d_x,   I_y,                   format='csc')
hodge_c_zx = sparse.kron(-Ax,                       diag_step_y_inv @ d_y, format='csc')
hodge_c_zy = sparse.kron( diag_step_x_inv @ d_x,   Ay,                    format='csc')

Z = None  # nulblok voor sparse.bmat

# ── Systeemmatrices [hx, hy, ezx, ezy] ──────────────────────────────────────
Mleft = sparse.bmat([
    [hodge_mu_xx/dt  + sigma_m_x/2,  Z,                               -hodge_c_xz/2,                        -hodge_c_xz/2                     ],
    [Z,                               hodge_mu_yy/dt  + sigma_m_y/2,  -hodge_c_yz/2,                        -hodge_c_yz/2                       ],
    [Z,                               -hodge_c_zy/2,                   hodge_eps_zz/dt + sigma_e_x/2,       Z                                  ],
    [hodge_c_zx/2,                    Z,                               Z,                                   hodge_eps_zz/dt + sigma_e_y/ 2     ],
], format='csc')

Mright = sparse.bmat([
    [hodge_mu_xx/dt  - sigma_m_x/2,  Z,                               hodge_c_xz/2,                       hodge_c_xz/2                      ],
    [Z,                               hodge_mu_yy/dt  - sigma_m_y/2,  hodge_c_yz/2,                       hodge_c_yz/2                      ],
    [Z,                               hodge_c_zy/2,                  hodge_eps_zz/dt - sigma_e_x/2,       Z                                  ],
    [-hodge_c_zx/2,                   Z,                               Z,                                   hodge_eps_zz/dt - sigma_e_y/2      ],
], format='csc')

# Mleft = sparse.bmat([[hodge_mu_xx/dt,None,-hodge_c_xz/2],
#                      [None,hodge_mu_yy/dt,-hodge_c_yz/2],
#                      [hodge_c_zx/2,hodge_c_zy/2,hodge_eps_zz/dt]])

# Mright = sparse.bmat([[hodge_mu_xx/dt,None,hodge_c_xz/2],
#                      [None,hodge_mu_yy/dt,hodge_c_yz/2],
#                      [-hodge_c_zx/2,-hodge_c_zy/2,hodge_eps_zz/dt]])


print("LU-factorisatie...")
Mleft_solver = sparse.linalg.factorized(Mleft)
print("Klaar.")


Grid size: 199 x 199, Time steps: 169
dx = 251153.5423 m, dt = 2.369536e-03 s
Lambda_min = 3767303.1347 m, points per wavelength = 15.0
N_PML = 49
PML matrices created
LU-factorisatie...
Klaar.


In [7]:

# ── Beginvelden [hx | hy | ezx | ezy] ───────────────────────────────────────
fields = np.zeros(4 * N)

def update(fields, current_time):
    rhs = Mright @ fields
    rhs[2*N + source_index] += get_source_value(current_time, dt)
    return Mleft_solver(rhs)

# ── Tijdlus ───────────────────────────────────────────────────────────────────
print("Simulatie gestart...")
snapshots     = {}
snap_interval = 10

for n in range(Nt):
    if n % 50 == 0:
        print(f"  Stap {n}/{Nt}")
    fields = update(fields, n * dt)
    if n % snap_interval == 0:
        ezx = fields[2*N:3*N].reshape((Nx, Ny))
        ezy = fields[3*N:4*N].reshape((Nx, Ny))
        snapshots[n] = (ezx + ezy).copy()

print("Simulatie klaar!")

# ── Animatie ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

snap_list = sorted(snapshots.keys())
all_ez = np.array(list(snapshots.values()))
vmax_abs = max(abs(all_ez.min()), abs(all_ez.max())) or 1.0

# Convert from meters to mm for display
x_mm = np.arange(Nx) * dx * 1e3
y_mm = np.arange(Ny) * dy * 1e3

im = ax.imshow(snapshots[snap_list[0]].T, cmap='RdBu', origin='lower',
               vmin=-vmax_abs, vmax=vmax_abs,
               extent=[0, Nx*dx*1e3, 0, Ny*dy*1e3])
plt.colorbar(im, ax=ax, label='$E_z$ [V/m]')

# PML-grenzen
for lim in [N_PML*dx*1e3, (Nx-N_PML)*dx*1e3]:
    ax.axvline(lim, color='gray', lw=0.8, ls='--', alpha=0.6)
for lim in [N_PML*dy*1e3, (Ny-N_PML)*dy*1e3]:
    ax.axhline(lim, color='gray', lw=0.8, ls='--', alpha=0.6)

# Bron en observatiepunt
ax.plot(ps_x*dx*1e3, ps_y*dy*1e3, 'r*', ms=10, label='Bron')
ax.plot(po_x*dx*1e3, po_y*dy*1e3, 'bx', ms=10, label='Observatie')

ax.legend(loc='upper right', fontsize=10)
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')
title = ax.set_title(f"Time = 0.00 s")

def animate(frame_idx):
    n = snap_list[frame_idx]
    im.set_data(snapshots[n].T)
    title.set_text(f"Time = {n*dt:.2e} s")
    return [im, title]

anim = animation.FuncAnimation(
    fig, animate, frames=len(snap_list),
    interval=50, blit=True, repeat=True
)

display(HTML(anim.to_jshtml()))
plt.close()

Simulatie gestart...
  Stap 0/169
  Stap 50/169
  Stap 100/169
  Stap 150/169
Simulatie klaar!
